# 08 - Crop Growth Model, LAI Assimilation & Uncertainty

The research-grade yield core:
1. **WOFOST-style wheat growth model** driven by IMD temperature/radiation.
2. **Ensemble Kalman Filter** assimilation of satellite-derived **LAI** (from S2) to correct the simulation for real stress.
3. **Uncertainty quantification**: conformal prediction intervals + ensemble spread on the final yield forecast.

In [ ]:
import sys
sys.path.append('..')
import numpy as np, matplotlib.pyplot as plt
from src import crop_model, uncertainty
np.random.seed(0)

## Daily drivers (Rabi season)
Replace these synthetic series with **IMD** daily Tmax/Tmean and radiation (`src.io_utils.load_imd_netcdf`). Terminal heat stress (Feb-Mar) is the key yield risk.

In [ ]:
days = np.arange(150)  # ~1 Nov -> end Mar
temps = 18 + 6*np.sin(2*np.pi*(days-30)/300) + np.random.normal(0,1,len(days))
temps[120:] += np.linspace(0, 10, len(days)-120)  # terminal heat ramp
rads = 14 + 4*np.sin(2*np.pi*(days)/200) + np.random.normal(0,0.8,len(days))
rads = np.clip(rads, 5, None)
plt.figure(figsize=(9,3)); plt.plot(days, temps, label='Tmean (C)'); plt.plot(days, rads, label='Radiation'); plt.legend(); plt.title('Daily drivers')

In [ ]:
model = crop_model.WheatGrowthModel()
traj = model.simulate(temps, rads)
open_loop_yield = model.yield_t_ha(traj[-1, 2])
print('Open-loop simulated yield: %.2f t/ha' % open_loop_yield)
plt.figure(figsize=(9,3)); plt.plot(traj[:,1]); plt.title('Simulated LAI (open loop)'); plt.xlabel('day'); plt.ylabel('LAI')

## EnKF assimilation of satellite LAI
Satellite LAI (from S2, e.g. SNAP biophysical or empirical NDVI->LAI) is assimilated at overpass dates. The filter corrects the canopy state and propagates a yield **ensemble**.

In [ ]:
obs_days = [30, 50, 70, 90, 110]
# 'True' field showing stress: lower LAI than open loop
true_model = crop_model.WheatGrowthModel(rue=2.3, harvest_index=0.40)
true_traj = true_model.simulate(temps, rads)
lai_obs = [true_traj[min(d, len(true_traj)-1), 1] for d in obs_days]

enkf = crop_model.EnKF(crop_model.WheatGrowthModel(), n_ens=80, lai_obs_std=0.25)
mean_traj, ens_yields = enkf.run(temps, rads, lai_obs, obs_days)
print('Assimilated mean yield: %.2f t/ha' % ens_yields.mean())

plt.figure(figsize=(9,4))
plt.plot(traj[:,1], label='open loop LAI')
plt.plot(mean_traj[:,1], label='EnKF assimilated LAI')
plt.scatter(obs_days, lai_obs, color='red', zorder=5, label='satellite LAI obs')
plt.legend(); plt.title('LAI: open loop vs EnKF assimilation'); plt.xlabel('day'); plt.ylabel('LAI')

## Uncertainty quantification
**(a)** Ensemble prediction interval from the EnKF yields. **(b)** Split-conformal interval calibrated on the district history.

In [ ]:
mean_y, lo, hi = uncertainty.ensemble_intervals(ens_yields, alpha=0.1)
print(f'EnKF yield: {mean_y:.2f} t/ha  90% PI [{lo:.2f}, {hi:.2f}]')
plt.figure(figsize=(7,3)); plt.hist(ens_yields, bins=20, color='seagreen', alpha=0.8)
plt.axvline(mean_y, color='k'); plt.axvline(lo, color='r', ls='--'); plt.axvline(hi, color='r', ls='--')
plt.title('EnKF yield ensemble distribution'); plt.xlabel('yield (t/ha)')

In [ ]:
import pandas as pd
hist = pd.read_csv('../data/sample/district_yield_history.csv')
# Toy point predictor: previous-year yield predicts current year
h = hist.sort_values(['district','year'])
h['pred'] = h.groupby('district')['yield_t_ha'].shift(1)
cal = h.dropna(subset=['pred'])
conf = uncertainty.SplitConformal(alpha=0.1)
q = conf.calibrate(cal['yield_t_ha'], cal['pred'])
lo_arr, hi_arr = conf.interval(cal['pred'].values)
coverage = ((cal['yield_t_ha'] >= lo_arr) & (cal['yield_t_ha'] <= hi_arr)).mean()
print(f'Conformal half-width q = {q:.3f} t/ha | empirical coverage = {coverage:.0%}')